In [1]:
pip install transformers datasets torch accelerate scikit-learn

   ---------------------------------------- 0.0/529.0 kB ? eta -:--:--
   ---------------------------------------- 529.0/529.0 kB 6.6 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [20]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from datasets import Dataset

from transformers import (

    DistilBertTokenizerFast,

    DistilBertForSequenceClassification,

    Trainer,

    TrainingArguments

)

from transformers import DataCollatorWithPadding

import torch

from sklearn.metrics import (
    accuracy_score,
    f1_score
)

In [3]:
df = pd.read_csv(
    r"D:\Guvi\Projects\ride_flow_ai\data\raw\nlp_reviews.csv"
)

print(df.head())

print(df.shape)

                                            feedback sentiment
0  a stirring , funny and finally transporting re...  positive
1  apparently reassembled from the cutting room f...  negative
2  they presume their audience wo n't sit still f...  negative
3  this is a visually stunning rumination on love...  positive
4  jonathan parker 's bartleby should have been t...  positive
(6920, 2)


In [4]:
df['label'] = df['sentiment'].map({

    'negative':0,

    'positive':1

})

In [5]:
train_df, test_df = train_test_split(

    df,

    test_size=0.2,

    random_state=42,

    stratify=df['label']
)

In [6]:
tokenizer = DistilBertTokenizerFast.from_pretrained(

    'distilbert-base-uncased'
)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

C:\Users\admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\admin\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [7]:
def tokenize(batch):

    return tokenizer(

        batch['feedback'],

        padding=True,

        truncation=True,

        max_length=128
    )

In [8]:
train_dataset = Dataset.from_pandas(
    train_df[['feedback','label']]
)

test_dataset = Dataset.from_pandas(
    test_df[['feedback','label']]
)

In [9]:
train_dataset = train_dataset.map(
    tokenize,
    batched=True
)

test_dataset = test_dataset.map(
    tokenize,
    batched=True
)

Map:   0%|          | 0/5536 [00:00<?, ? examples/s]

Map:   0%|          | 0/1384 [00:00<?, ? examples/s]

train_dataset.set_format(

    type='torch',

    columns=[
        'input_ids',
        'attention_mask',
        'label'
    ]
)

test_dataset.set_format(

    type='torch',

    columns=[
        'input_ids',
        'attention_mask',
        'label'
    ]
)

In [23]:
model = DistilBertForSequenceClassification.from_pretrained(

    'distilbert-base-uncased',

    num_labels=2
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [24]:
def compute_metrics(pred):

    labels = pred.label_ids

    preds = pred.predictions.argmax(-1)

    acc = accuracy_score(
        labels,
        preds
    )

    f1 = f1_score(
        labels,
        preds
    )

    return {

        'accuracy':acc,

        'f1':f1
    }

In [25]:
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

In [26]:
training_args = TrainingArguments(

    output_dir=r"D:\Guvi\Projects\ride_flow_ai\models\distilbert_results",

    learning_rate=2e-5,

    per_device_train_batch_size=8,

    per_device_eval_batch_size=8,

    num_train_epochs=2,

    weight_decay=0.01,

    logging_dir="../logs"
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [29]:
trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=test_dataset,

    data_collator=data_collator,

    compute_metrics=compute_metrics
)

In [30]:
trainer.train()

Step,Training Loss
500,0.403913
1000,0.269740


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

C:\Users\admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1384, training_loss=0.3005512380875604, metrics={'train_runtime': 7193.5242, 'train_samples_per_second': 1.539, 'train_steps_per_second': 0.192, 'total_flos': 190510887101472.0, 'train_loss': 0.3005512380875604, 'epoch': 2.0})

In [31]:
results = trainer.evaluate()

print(results)

C:\Users\admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


RuntimeError: on_train_begin must be called before on_evaluate

In [32]:
predictions = trainer.predict(test_dataset)

print(predictions.metrics)

C:\Users\admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


{'test_loss': 0.36865273118019104, 'test_accuracy': 0.9067919075144508, 'test_f1': 0.9110957960027567, 'test_runtime': 264.5423, 'test_samples_per_second': 5.232, 'test_steps_per_second': 0.654}


In [33]:
model.save_pretrained(
    r"D:\Guvi\Projects\ride_flow_ai\models\distilbert_sentiment_model"
)

tokenizer.save_pretrained(
    r"D:\Guvi\Projects\ride_flow_ai\models\distilbert_sentiment_model"
)

print("DistilBERT Model Saved")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

DistilBERT Model Saved
